In [4]:
import re
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urljoin, urlparse, unquote
import requests

flare_url = "https://ovsa.njit.edu/eovsadata/api/flare.php?date=2026-01-18&flare_id=20260118175000"
access_link = input("Paste the EOVSA access link from your email: ").strip()
output_dir = Path("eovsa_fits_20260118175000")
output_dir.mkdir(exist_ok=True)

Paste the EOVSA access link from your email:  https://ovsa.njit.edu/eovsadata/api/verify.php?token=3fe722b4a0f5f81b7bf06e5d092ee907c7c43e771ccc70978e368a70c0d7903a


In [2]:
class LinkCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.hrefs = []
    def handle_starttag(self, tag, attrs):
        if tag == "a":
            href = dict(attrs).get("href")
            if href:
                self.hrefs.append(href)

In [ ]:
"/home/mnedal/data/2026-01-18/EOVSA/"

In [5]:
session = requests.Session()

# Redeem the emailed link; this should set the EOVSADATA session cookie.
login = session.get(access_link, timeout=60, allow_redirects=True)
login.raise_for_status()

page = session.get(flare_url, timeout=60, allow_redirects=False)
if page.is_redirect or page.status_code != 200:
    raise RuntimeError(
        "The access link did not create an authorized session. "
        "Request a new link and paste it here before opening it anywhere else."
    )

parser = LinkCollector()
parser.feed(page.text)

fits_urls = sorted({
    urljoin(flare_url, href)
    for href in parser.hrefs
    if re.search(r"\.fits(?:$|[?&#])", href, re.I)
    or re.search(r"(?:format|ext)=fits(?:$|[&#])", href, re.I)
})

if not fits_urls:
    raise RuntimeError("No FITS links were found on the authorized flare page.")

for n, url in enumerate(fits_urls, 1):
    response = session.get(url, stream=True, timeout=180)
    response.raise_for_status()

    if "text/html" in response.headers.get("Content-Type", "").lower():
        raise RuntimeError(f"Authorization failed while downloading:\n{url}")

    cd = response.headers.get("Content-Disposition", "")
    match = re.search(r'filename[^=]*=(?:UTF-8\'\')?["\']?([^"\';]+)', cd, re.I)
    filename = unquote(match.group(1)) if match else Path(urlparse(response.url).path).name
    if not filename.lower().endswith(".fits"):
        filename = f"flare_{n:03d}.fits"

    destination = output_dir / filename
    with destination.open("wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    print(f"Downloaded: {destination}")

print(f"\nDone: {len(fits_urls)} FITS files saved to {output_dir.resolve()}")

HTTPError: 403 Client Error: Forbidden for url: https://ovsa.njit.edu/eovsadata/api/verify.php?token=3fe722b4a0f5f81b7bf06e5d092ee907c7c43e771ccc70978e368a70c0d7903a